# Voice RAG Assistant — Cloud GPU Backend

Run this notebook instead of your local backend to get **T4 GPU** acceleration for Whisper, Reranker and **Kokoro TTS**.

**Setup (one-time):**
1. Runtime → Change runtime type → **T4 GPU**
2. Click the key icon (Secrets) on the left and add:
   - `GEMINI_API_KEY`
   - `DEEPSEEK_API_KEY`
   - `NGROK_TOKEN` — free from https://dashboard.ngrok.com/get-started/your-authtoken
3. Run all cells top to bottom.
4. Copy the **PUBLIC BACKEND URL** printed by the last cell into your local `frontend/.env` as `VITE_BACKEND_URL=<url>`, then restart `npm run dev`.

> Free Colab sessions expire (idle timeout / max ~12h). If the runtime dies, re-run all cells and update the URL.

In [ ]:
# 1. Confirm GPU is attached
import torch
if torch.cuda.is_available():
    print("GPU OK:", torch.cuda.get_device_name(0))
else:
    print("NO GPU — go to Runtime > Change runtime type > T4 GPU, then re-run")

In [ ]:
# 2. Upload backend_colab.zip when prompted
from google.colab import files
import zipfile, os

up = files.upload()
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall("/content")
assert os.path.isdir("/content/backend/app"), "unzip failed — check the zip"
print("backend extracted:", sorted(os.listdir("/content/backend")))

In [ ]:
# 3. Install dependencies (Colab already has CUDA torch — skip re-downloading it)
!grep -viE "^(torch|torchvision)" /content/backend/requirements.txt > /tmp/req.txt
!pip -q install -r /tmp/req.txt
!pip -q install pyngrok
print("deps installed")

In [ ]:
# 4. Load API keys from Colab Secrets (key icon on the left)
from google.colab import userdata
import os

for k in ["GEMINI_API_KEY", "DEEPSEEK_API_KEY", "NGROK_TOKEN"]:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print("WARNING: secret not set:", k)

# allow the vite dev proxy to reach this instance
os.environ["CORS_ORIGINS"] = "*"
print("keys loaded")

In [ ]:
# 5. Start the backend and expose it via ngrok
import subprocess, time, requests

srv = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/backend",
)

for _ in range(80):
    try:
        if requests.get("http://localhost:8000/api/v1/health", timeout=3).status_code == 200:
            print("backend is UP (models loaded)")
            break
    except Exception:
        pass
    time.sleep(3)
else:
    print("backend did not become healthy — check cell output above")

from pyngrok import ngrok, conf
token = os.environ.get("NGROK_TOKEN")
if not token:
    raise SystemExit("Set the NGROK_TOKEN secret, then re-run this cell")
conf.get_default().auth_token = token
ngrok.kill()
url = ngrok.connect(8000)
print()
print("=" * 60)
print("PUBLIC BACKEND URL:", url.public_url)
print("=" * 60)
print("Paste this into local frontend/.env as:")
print(f"VITE_BACKEND_URL={url.public_url}")
print("then restart: npm run dev")